# NLP

- [NLP Tasks and Language Models](#NLP-Tasks-and-Language-Models)
  - [NLP task landscape](#NLP-task-landscape)
  - [Language models and perplexity](#Language-models-and-perplexity)
  - [Topic modelling](#Topic-modelling)
- [Traditional NLP](#Traditional-NLP)
  - [Pipeline overview](#Pipeline-overview)
  - [Tokenization and word embeddings](#Tokenization-and-word-embeddings)
  - [POS tagging and Named Entity Recognition (NER)](#POS-tagging-and-Named-Entity-Recognition-(NER))
- [Sequence Models: RNN, LSTM, GRU](#Sequence-Models:-RNN,-LSTM,-GRU)
  - [Why sequential models](#Why-sequential-models)
  - [RNN core mechanics, worked by hand](#RNN-core-mechanics,-worked-by-hand)
  - [The vanishing gradient problem](#The-vanishing-gradient-problem)
  - [LSTM: fixing vanishing gradients with an additive cell state](#LSTM:-fixing-vanishing-gradients-with-an-additive-cell-state)
  - [GRU, briefly](#GRU,-briefly)
  - [Seq2seq and attention](#Seq2seq-and-attention)
  - [Long documents and multilingual text](#Long-documents-and-multilingual-text)

### NLP Tasks and Language Models

#### NLP task landscape

Text classification/sentiment; sequence labelling (NER, POS); QA (extractive vs generative); summarization (extractive/abstractive); machine translation; retrieval/semantic similarity; generation. Modern default: pretrained transformer fine-tuned or prompted; metrics differ (F1, EM, ROUGE, BLEU).

#### Language models and perplexity

LM = P(next token | context). N-gram: counts + smoothing, fixed window, sparse. Neural LM: learned embeddings, long context (RNN/transformer). Perplexity=exp(avg negative log-likelihood) ≈ effective branching factor; lower is better; only comparable with the same tokenizer/vocab/test set.

#### Topic modelling

LDA: generative; documents = mixtures of topics, topics = word distributions (Dirichlet priors; Gibbs/variational); bag-of-words, must pick K, interpretable topic words. Embedding-based (BERTopic): embed docs → UMAP → HDBSCAN → c-TF-IDF; captures semantics/short text, auto-K. Evaluate with topic coherence.

### Traditional NLP

Two sections: traditional NLP (tokenization through word embeddings, pre-neural-network techniques), then RNN/LSTM (sequential neural models). Text-feature techniques (stemming/lemmatization, n-grams, TF-IDF) are in `04-data-prep.ipynb`. Transformers and LLMs are out of scope here, covered separately in `05-tokens-arch.ipynb`.

#### Pipeline overview

raw text -> tokenization -> normalization (lowercase, stemming/lemmatization) -> stopword removal -> numeric representation (bag-of-words, TF-IDF, or embeddings) -> features for a model. Every step below is one stage of this pipeline.

#### Tokenization and word embeddings

Covered in `05-tokens-arch.ipynb`: real tokenizers (tiktoken) and embeddings (Word2Vec, FastText, BERT). This notebook covers POS tagging and NER, then RNN/LSTM.

#### POS tagging and Named Entity Recognition (NER)

POS (part-of-speech) tagging: label each token with its grammatical role, noun/verb/adjective/etc. Useful as a feature itself, or as a preprocessing filter (e.g. keep only nouns and verbs before building a TF-IDF vocabulary, dropping function words like "the"/"a" more precisely than a fixed stopword list).

NER: identify and classify spans of text into predefined categories, PERSON, ORGANIZATION, LOCATION, MONEY, DATE. Directly relevant to this project's fraud narratives, a rule-based or statistical NER model could extract "IRS" as an ORGANIZATION and "$10,000" as a MONEY amount BEFORE any LLM involvement at all, this is exactly the kind of structured extraction the LLM-based FEATURE_SCHEMA in the fraud project performs, NER was the pre-LLM way to attempt the same task, with a fixed set of entity categories instead of an LLM's flexible schema, and no ability to extract something as nuanced as "impersonated_entity" (a role, not a named entity in the traditional sense).

In [ ]:
import re

# real from-scratch rule-based NER, a toy version of what spaCy's statistical NER does at scale
def simple_ner(text):
    entities = []
    for match in re.finditer(r"\$[\d,]+", text):
        entities.append((match.group(), "MONEY"))
    for org in ["IRS", "FBI", "Social Security"]:
        if org in text:
            entities.append((org, "ORGANIZATION"))
    return entities

text = "Someone claiming to be from the IRS demanded $10,000 in gift cards."
print("entities found:", simple_ner(text))

print("\nthis is a fixed-keyword/regex rule, not a trained model, it would miss any")
print("organization not on the hardcoded list, exactly why real NER uses a statistical")
print("or neural model trained on labeled entity spans instead. spaCy is the common")
print("library for the production version: pip install spacy (not installed here)")

### Sequence Models: RNN, LSTM, GRU

#### Why sequential models

Everything above (TF-IDF, n-grams, even Word2Vec) either ignores word order or captures only a small fixed window of it. RNNs process a sequence step by step, maintaining a HIDDEN STATE that carries information forward from every previous step, in principle capturing dependencies across an entire sequence, not just a fixed n-gram window.

#### RNN core mechanics, worked by hand

Formula: h_t = tanh(W_h * h_(t-1) + W_x * x_t + b)

Toy setup: input sequence x = [1, 2] (2 timesteps, scalar input), initial hidden state h_0=0, weights W_h=0.5, W_x=1.0, bias=0.
```
h_1 = tanh(W_h*h_0 + W_x*x_1 + b) = tanh(0.5*0 + 1.0*1 + 0) = tanh(1.0) = 0.7616
h_2 = tanh(W_h*h_1 + W_x*x_2 + b) = tanh(0.5*0.7616 + 1.0*2 + 0) = tanh(2.3808) = 0.9827
```
h_2 depends on h_1, which depends on x_1, so h_2 carries information from BOTH x_1 and x_2, even though the formula at step 2 only directly references h_1 and x_2. This chaining is the entire mechanism, each hidden state is a compressed summary of everything seen so far.

In [ ]:
import numpy as np

def rnn_step(h_prev, x_t, W_h=0.5, W_x=1.0, b=0.0):
    return np.tanh(W_h * h_prev + W_x * x_t + b)

h = 0.0
for t, x_t in enumerate([1, 2], start=1):
    h = rnn_step(h, x_t)
    print(f"h_{t} = {h:.4f}")

#### The vanishing gradient problem

Training an RNN (backpropagation through time) requires the gradient of the loss at the final timestep to flow backward through every intermediate timestep, and at each step backward it gets multiplied by W_h and by tanh's derivative (which is always <=1, and often much less than 1 when the pre-activation is large, tanh saturates).

Worked example: suppose that combined per-step multiplier (W_h * tanh'(z)) averages around 0.5 across timesteps. To reach a weight update from a loss computed 9 timesteps later, the gradient signal gets multiplied by roughly 0.5 nine times:
```
0.5^9 = 0.00195
```
The gradient contribution from an early timestep is scaled down to under 0.2% of its original size by the time it reaches a weight update, effectively zero. The network cannot learn dependencies that span more than a handful of timesteps, the gradient signal needed to learn them has vanished before it gets there. (The reverse failure mode, exploding gradients, happens when that multiplier is consistently greater than 1, gradients blow up instead of vanishing, usually handled with gradient clipping.)

#### LSTM: fixing vanishing gradients with an additive cell state

Three gates, plus a cell state that runs alongside the hidden state:
```
forget gate:  f_t = sigmoid(...)   how much of the OLD cell state to keep
input gate:   i_t = sigmoid(...)   how much of the NEW candidate info to add
candidate:    c~_t = tanh(...)     the new information itself
cell update:  c_t = f_t * c_(t-1) + i_t * c~_t     <- ADDITIVE, not multiplicative-through-tanh
output gate:  o_t = sigmoid(...)
hidden state: h_t = o_t * tanh(c_t)
```
Worked example, one timestep: f_t=0.9 (mostly keep the old cell state), i_t=0.3, c~_t=0.8, c_(t-1)=2.0, o_t=0.7.
```
c_t = 0.9*2.0 + 0.3*0.8 = 1.8 + 0.24 = 2.04
h_t = 0.7 * tanh(2.04) = 0.7 * 0.9668 = 0.677
```
Why this fixes the vanishing gradient: the gradient of c_t with respect to c_(t-1) is just f_t (close to 1 whenever the forget gate is open), not squashed through a repeated tanh multiplication the way the plain RNN's hidden state update was. Information (and gradient) can flow across many timesteps largely unchanged whenever the forget gate stays open, an additive highway instead of a repeatedly-multiplied one.

#### GRU, briefly

Simplified LSTM: merges the forget and input gates into a single "update gate," and combines the cell state and hidden state into one. Fewer parameters, faster to train, similar performance to LSTM on many tasks, no strict rule for which wins on a given problem, worth trying both.

In [ ]:
import torch
import torch.nn as nn

seq = torch.tensor([[[1.0], [2.0]]])  # shape (batch=1, seq_len=2, input_size=1)

rnn = nn.RNN(input_size=1, hidden_size=1, batch_first=True)
lstm = nn.LSTM(input_size=1, hidden_size=1, batch_first=True)

rnn_out, rnn_hidden = rnn(seq)
lstm_out, (lstm_hidden, lstm_cell) = lstm(seq)

print("RNN hidden states per timestep:", rnn_out.detach().numpy().flatten())
print("LSTM hidden states per timestep:", lstm_out.detach().numpy().flatten())
print("(random-initialized weights here, won't match the hand-worked numbers above, same mechanism though)")

#### Seq2seq and attention

Encoder compresses variable-length input into a fixed context vector, decoder generates variable-length output from it — handles input/output length mismatch (translation, summarization). Limitation: single fixed-size context vector bottlenecks long inputs — attention mechanism fixes this by letting the decoder look back at all encoder states, not just the final one.

At each step the decoder computes weights over all encoder states (Bahdanau additive, Luong multiplicative) and builds a context vector, removing the single-vector bottleneck. Vanilla encoder–decoder squeezes the whole input into one fixed vector (bottleneck; degrades on long sequences). Attention: decoder scores all encoder states (Bahdanau additive, Luong dot/multiplicative) → softmax weights → context vector per step ⇒ soft alignment; foundation for Transformers.

#### Long documents and multilingual text

Long: 512-token limit ⇒ truncate, chunk + aggregate, hierarchical, or long-context models (Longformer/BigBird sparse attention, RoPE-scaled LLMs). Multilingual: mBERT/XLM-R share a subword vocab ⇒ cross-lingual transfer; watch low-resource imbalance, tokenization fertility, evaluate per language.